# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant package
dataset = mlc.Dataset(croissant_url)

# Access the dataset's metadata
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the schema to find all top-level record sets. Each record set, field, and column is referenced by its `@id` for consistency with Croissant best practices.

In [ ]:
# List all record sets defined in Croissant metadata
print('Record sets in this dataset:')
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name')}")
    print('  Fields:')
    for field in rs['field']:
        print(f"    - @id: {field['@id']}, name: {field.get('name')}, dataType: {field.get('dataType')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** All entity identifiers are referenced by their Croissant `@id`.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print('Available record set identifiers:')
for idx, rsid in enumerate(record_set_ids):
    print(f"{idx+1}: {rsid}")

# For demonstration, select the first record set
selected_record_set_id = record_set_ids[0]
print(f"\nExtracting data from record set: {selected_record_set_id}")

# Load records into DataFrames for each record set
dataframes = {}
for record_set_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df

print(f"\nColumns (fields) in record set {selected_record_set_id}:")
print(dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering records by field value, normalizing numeric variables, and grouping by key attributes.

Entities are referenced by their `@id`.

In [ ]:
# Identify a numeric field for analysis (by @id)
# Replace this with the actual numeric field @id for your dataset; here we search for 'Age' or other Integer/Float fields

fields_info = {field['@id']: field for field in dataset.record_sets[0]['field']}
numerics = [fid for fid, f in fields_info.items() if f.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']]
print('Numeric fields found:', numerics)
if numerics:
    numeric_field_id = numerics[0] # Use the first numeric field
else:
    raise ValueError('No numeric field found in the first record set.')

# For grouping, pick a categorical field (commonly 'Sex', 'MSI status', etc.)
# Here, select the first non-numeric field after skipping 'name', or set by inspection
categoricals = [fid for fid, f in fields_info.items() if f.get('dataType') not in ['schema:Integer', 'schema:Float', 'schema:Number'] and fid != 'name']
if categoricals:
    group_field_id = categoricals[0]
else:
    group_field_id = None
print('Categorical field for grouping:', group_field_id)

df = dataframes[selected_record_set_id]
print(f"\nAvailable columns in DataFrame: {df.columns.tolist()}")

if numeric_field_id in df.columns:
    # Convert to numeric, handle coercion if necessary
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Use mean as threshold for illustration
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped statistics
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df)
else:
    print(f"Numeric field {numeric_field_id} is not present in dataframe columns.")

## 5. Visualization
Visualize distributions of the selected numeric field and grouped means by categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to load dataset metadata, inspect record sets, load records into DataFrames, and perform basic EDA and visualization. All dataset entities including record sets, fields, and columns were referenced by their Croissant `@id`, ensuring reproducibility and schema-aligned analysis.

Explore the [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) further by examining additional fields and record sets using their `@id`.